# LangGraph Persistence, Time Travel & Fault Tolerance

This notebook demonstrates three powerful features of LangGraph:
1. **Persistence** — saving graph state across runs using checkpointers
2. **Time Travel** — inspecting and replaying past states
3. **Fault Tolerance** — resuming a crashed/interrupted graph from the last saved checkpoint

---

## Part 1: Setup & Imports

In [2]:
import time

from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import InMemorySaver

from dotenv import load_dotenv

load_dotenv()

c:\Dexter\github\AsifAmir\LangGraph\venv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


True

## Part 2: LLM Initialization

In [3]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0.7)

## Part 3: Define the Graph State

In LangGraph, **state** is a typed dictionary shared across all nodes.
Each node reads from the state and returns a partial update to it.

In [4]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

## Part 4: Define the Graph Nodes

Each **node** is a Python function that:
- Accepts the current `state` as input
- Returns a dictionary with the fields it wants to update

In [5]:
def generate_joke(state: JokeState):

    # Construct a simple prompt using the topic stored in the current state
    prompt = f'generate a joke on the topic {state["topic"]}'

    # Invoke the LLM and extract the text content from the response
    response = llm.invoke(prompt).content

    # Return a partial state update — only `joke` is updated here
    return {'joke': response}

In [6]:
def generate_explanation(state: JokeState):

    # Use the joke from the previous node to construct the explanation prompt
    prompt = f'write an explanation for the joke - {state["joke"]}'

    # Invoke the LLM and extract the text content
    response = llm.invoke(prompt).content

    # Return a partial state update — only `explanation` is updated here
    return {'explanation': response}

## Part 5: Build & Compile the Graph

Here we wire nodes together, define the execution flow (edges), and attach a **checkpointer**.

The `InMemorySaver` checkpointer automatically saves a snapshot of the full state
after every node execution — enabling persistence, time travel, and fault tolerance.

In [7]:
# Step 1: Create a new StateGraph using our JokeState schema
graph = StateGraph(JokeState)

# Step 2: Register nodes — name them so edges can reference them
graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

# Step 3: Define execution order using directed edges
# START → generate_joke → generate_explanation → END
graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

# Step 4: Create an in-memory checkpointer
# InMemorySaver stores state snapshots in RAM (not persisted to disk).
# Each snapshot is keyed by (thread_id, checkpoint_id) so multiple
# independent conversations can be tracked simultaneously.
checkpointer = InMemorySaver() # create an in-memory checkpointer to save the state of the conversation

# Step 5: Compile the graph with the checkpointer attached
# After compilation, the graph becomes a runnable `CompiledGraph` (called `workflow` here)
workflow = graph.compile(checkpointer=checkpointer)

In [ ]:
# Displaying the compiled `workflow` object renders a visual diagram of the graph
workflow

## Part 6: Running the Graph with Thread IDs

A **thread** is an isolated conversation/session. Each thread has its own state history.
The `thread_id` in the config is what keeps different sessions separate.

In [8]:
# Define config for thread "1" — this uniquely identifies this session
config1 = {"configurable": {"thread_id": "1"}}

# Invoke the graph: sets topic='pizza', runs both nodes, and returns final state
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza get a job at the bakery?\n\nBecause it kneaded the dough!',
 'explanation': 'Here\'s the explanation for the pizza joke:\n\nThis joke is a **pun**, which means it relies on a word or phrase having two different meanings, or sounding like another word or phrase.\n\nLet\'s break it down:\n\n*   **"Kneaded the dough"** is the key phrase.\n    *   **Literal meaning (related to pizza):** When you make pizza, you have to **knead** the **dough** (flour, water, yeast, etc.) to develop gluten, making it stretchy and elastic. So, a pizza *literally* needs dough to be made.\n    *   **Figurative meaning (related to getting a job):** In everyday language, "to need" means to require or want something. If someone "needs the dough," it means they need money (slang for money is "dough").\n\nThe humor comes from the unexpected switch between these two meanings. The joke sets you up to think about the pizza\'s job at a bakery (where dough is also used), and

In [ ]:
# Inspect the Current (Latest) State of Thread 1
workflow.get_state(config1)

In [9]:
# View the Full State History of Thread 1
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a job at the bakery?\n\nBecause it kneaded the dough!', 'explanation': 'Here\'s the explanation for the pizza joke:\n\nThis joke is a **pun**, which means it relies on a word or phrase having two different meanings, or sounding like another word or phrase.\n\nLet\'s break it down:\n\n*   **"Kneaded the dough"** is the key phrase.\n    *   **Literal meaning (related to pizza):** When you make pizza, you have to **knead** the **dough** (flour, water, yeast, etc.) to develop gluten, making it stretchy and elastic. So, a pizza *literally* needs dough to be made.\n    *   **Figurative meaning (related to getting a job):** In everyday language, "to need" means to require or want something. If someone "needs the dough," it means they need money (slang for money is "dough").\n\nThe humor comes from the unexpected switch between these two meanings. The joke sets you up to think about the pizza\'s job at a bakery (where doug

In [10]:
# Define config for thread "2" — Thread 2 with topic "pasta"
config2 = {"configurable": {"thread_id": "2"}}

# Run the same graph with a different topic under a new thread
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti break up with the linguine?\n\nBecause he felt like she was always winding up in his business!',
 'explanation': 'Here\'s the explanation for the joke:\n\nThe humor in this joke comes from a **pun** that plays on the literal shape of the pasta and a common idiom.\n\n*   **Literal Meaning (Pasta):** Spaghetti and linguine are both types of long, thin pasta. Spaghetti is round in cross-section, while linguine is flat and slightly wider. The idea of them "winding up" in each other\'s business refers to their physical shapes intertwining or getting tangled together.\n\n*   **Figurative Meaning (Idiom):** The idiom "winding up in someone\'s business" means to get involved in their affairs, often unnecessarily or intrusively. It implies meddling or being too nosy.\n\n**The Joke\'s Punchline:**\n\nThe spaghetti "breaks up" with the linguine because he feels she\'s always "winding up in his business." This is funny because:\n\n1.  **The Double

In [11]:
# Verify Thread 1 State is Unchanged by Thread 2's Run
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a job at the bakery?\n\nBecause it kneaded the dough!', 'explanation': 'Here\'s the explanation for the pizza joke:\n\nThis joke is a **pun**, which means it relies on a word or phrase having two different meanings, or sounding like another word or phrase.\n\nLet\'s break it down:\n\n*   **"Kneaded the dough"** is the key phrase.\n    *   **Literal meaning (related to pizza):** When you make pizza, you have to **knead** the **dough** (flour, water, yeast, etc.) to develop gluten, making it stretchy and elastic. So, a pizza *literally* needs dough to be made.\n    *   **Figurative meaning (related to getting a job):** In everyday language, "to need" means to require or want something. If someone "needs the dough," it means they need money (slang for money is "dough").\n\nThe humor comes from the unexpected switch between these two meanings. The joke sets you up to think about the pizza\'s job at a bakery (where dough

In [12]:
# Confirm Thread 1 History is Still Isolated and Unaffected by Thread 2's Run
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a job at the bakery?\n\nBecause it kneaded the dough!', 'explanation': 'Here\'s the explanation for the pizza joke:\n\nThis joke is a **pun**, which means it relies on a word or phrase having two different meanings, or sounding like another word or phrase.\n\nLet\'s break it down:\n\n*   **"Kneaded the dough"** is the key phrase.\n    *   **Literal meaning (related to pizza):** When you make pizza, you have to **knead** the **dough** (flour, water, yeast, etc.) to develop gluten, making it stretchy and elastic. So, a pizza *literally* needs dough to be made.\n    *   **Figurative meaning (related to getting a job):** In everyday language, "to need" means to require or want something. If someone "needs the dough," it means they need money (slang for money is "dough").\n\nThe humor comes from the unexpected switch between these two meanings. The joke sets you up to think about the pizza\'s job at a bakery (where doug

---
## Part 7: Time Travel

**Time Travel** lets you go back to any previous checkpoint and either:
- **Inspect** — read what the state looked like at that moment
- **Replay** — re-run the graph from that historical point forward

This is powerful for debugging, branching experiments, or "what if" scenarios.

> **How to get a checkpoint_id?**  
> Run `get_state_history(config)` and copy a `checkpoint_id` from any past snapshot.

In [13]:
# Inspect a Specific Historical Checkpoint (Read-Only)
# NOTE: Replace the checkpoint_id below with one from your own run
# (copy it from the output of Cell 10 / Cell 13).

workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f15289e-b3d8-689b-8000-367112d1a87d"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f15289e-b3d8-689b-8000-367112d1a87d'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-05-18T07:19:33.587164+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f15289e-b3d6-6138-bfff-7ef4bfbf024d'}}, tasks=(PregelTask(id='1ad3c79d-63ba-35f3-c5f6-c3afaa79e2e7', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the pizza get a job at the bakery?\n\nBecause it kneaded the dough!'}),), interrupts=())

In [14]:
# Replay the Graph from a Historical Checkpoint
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f15289e-b3d8-689b-8000-367112d1a87d"}})

{'topic': 'pizza',
 'joke': 'Why did the pizza maker get fired?\n\nBecause he kept rolling out of control!',
 'explanation': 'This is a classic pun-based joke! Here\'s the explanation:\n\n*   **The Setup:** "Why did the pizza maker get fired?" This sets up an expectation for a reason related to a job, likely something to do with making pizza.\n\n*   **The Punchline:** "Because he kept rolling out of control!"\n\n*   **The Double Meaning (the pun):**\n    *   **Literal Meaning (related to pizza making):** Pizza dough is often "rolled out" using a rolling pin to make it flat and round. If the pizza maker was doing this poorly, perhaps unevenly, too thin, or even "out of control" in terms of technique, it could be a reason for getting fired.\n    *   **Figurative Meaning (related to behavior):** The phrase "out of control" also means behaving erratically, impulsively, or in a way that is difficult to manage. It implies a lack of self-discipline or recklessness.\n\n*   **The Humor:** The j

In [15]:
# Confirm Time Travel Created New Checkpoints

list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza maker get fired?\n\nBecause he kept rolling out of control!', 'explanation': 'This is a classic pun-based joke! Here\'s the explanation:\n\n*   **The Setup:** "Why did the pizza maker get fired?" This sets up an expectation for a reason related to a job, likely something to do with making pizza.\n\n*   **The Punchline:** "Because he kept rolling out of control!"\n\n*   **The Double Meaning (the pun):**\n    *   **Literal Meaning (related to pizza making):** Pizza dough is often "rolled out" using a rolling pin to make it flat and round. If the pizza maker was doing this poorly, perhaps unevenly, too thin, or even "out of control" in terms of technique, it could be a reason for getting fired.\n    *   **Figurative Meaning (related to behavior):** The phrase "out of control" also means behaving erratically, impulsively, or in a way that is difficult to manage. It implies a lack of self-discipline or recklessness.\n\n*   

---
### Part 7b: Updating State (Manual State Injection)

You can also **manually modify** the values in a historical checkpoint
before replaying from it — like editing a "save file" in a video game.
This creates a new forked checkpoint with the updated values.

In [16]:
# Manually Update State at a Specific Checkpoint
# Override the topic field in the saved state
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f15289e-b3d8-689b-8000-367112d1a87d", "checkpoint_ns": ""}}, {'topic':'Ramen'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1528aa-0929-6e63-8001-b1a8c8eef620'}}

In [17]:
# Verify the Updated State Appears in History
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Ramen'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1528aa-0929-6e63-8001-b1a8c8eef620'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-05-18T07:24:37.812387+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f15289e-b3d8-689b-8000-367112d1a87d'}}, tasks=(PregelTask(id='830cfe09-4043-8888-4f75-7035518a4a44', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza maker get fired?\n\nBecause he kept rolling out of control!', 'explanation': 'This is a classic pun-based joke! Here\'s the explanation:\n\n*   **The Setup:** "Why did the pizza maker get fired?" This sets up an expectation for a reason related to a job, likely something to do with making pizza.\n\n*   **The Pun

In [18]:
# CELL 19: Replay Graph from the Updated (Ramen) Checkpoint
# NOTE: Replace the checkpoint_id below with the ID of the
# newly created 'Ramen' checkpoint (from the output of Cell 18).

workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f1528aa-0929-6e63-8001-b1a8c8eef620"}})

{'topic': 'Ramen',
 'joke': 'Why did the ramen noodle break up with the broth?\n\nBecause he felt like he was just getting *soup*-erficial love!',
 'explanation': 'This joke plays on a **pun** and a common idiom. Here\'s the breakdown:\n\n*   **The Setup:** "Why did the ramen noodle break up with the broth?" This sets up a classic joke structure, leading to an unexpected answer. Ramen noodles and broth are naturally paired, so the idea of them breaking up is already humorous.\n\n*   **The Punchline:** "Because he felt like he was just getting *soup*-erficial love!"\n\n*   **The Pun:** The key word here is "soup-erficial."\n    *   **"Soup"** is a direct reference to the ramen\'s broth.\n    *   **"Superficial"** is a word that means "concerned only with what is obvious or apparent; not deep or profound." It describes a love or relationship that lacks depth, sincerity, or genuine connection.\n\n*   **The Idiom:** The joke uses the phrase "superficial love." The ramen noodle is "breaking

In [19]:
# View Final History After All Time Travel Operations
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Ramen', 'joke': 'Why did the ramen noodle break up with the broth?\n\nBecause he felt like he was just getting *soup*-erficial love!', 'explanation': 'This joke plays on a **pun** and a common idiom. Here\'s the breakdown:\n\n*   **The Setup:** "Why did the ramen noodle break up with the broth?" This sets up a classic joke structure, leading to an unexpected answer. Ramen noodles and broth are naturally paired, so the idea of them breaking up is already humorous.\n\n*   **The Punchline:** "Because he felt like he was just getting *soup*-erficial love!"\n\n*   **The Pun:** The key word here is "soup-erficial."\n    *   **"Soup"** is a direct reference to the ramen\'s broth.\n    *   **"Superficial"** is a word that means "concerned only with what is obvious or apparent; not deep or profound." It describes a love or relationship that lacks depth, sincerity, or genuine connection.\n\n*   **The Idiom:** The joke uses the phrase "superficial love." The ramen

---
## Fault Tolerance

**Fault Tolerance** means the graph can survive crashes, kernel interruptions, or network failures
and **resume from where it left off** rather than starting over.

This works because the checkpointer saves state **after each node completes**.
If a node crashes mid-run, the previous node's output is still safely stored.

In this example we simulate a crash using `time.sleep(1000)` + a Kernel Interrupt.

In [20]:
# 1. Define CrashState — State Schema for Fault Tolerance Demo
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [21]:
# 2. Define steps

# step_1 — runs quickly and successfully
# step_2 — simulates a crash by sleeping for 1000 seconds;
#           the user must manually interrupt the kernel (STOP button)
#           to simulate a real-world failure (timeout, network error, etc.)
# step_3 — intended to run after step_2; won't be reached in the first run

def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [22]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Simulate a Crash (First Run)
# ─────────────────────────────────────────────────────────────
# Run the graph. It will:
#   1. Execute step_1 successfully → checkpoint saved ✅
#   2. Enter step_2 and hang at time.sleep(1000)
#   3. YOU manually stop the kernel (click the STOP/Interrupt button)
#   4. KeyboardInterrupt is caught and printed
#
# After the interrupt, the checkpoint from step_1 is still safely
# stored in the InMemorySaver — nothing is lost.

try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 25: Resume from Last Checkpoint (Fault Tolerance in Action)
# ─────────────────────────────────────────────────────────────
# Calling invoke with `None` (no new input) and the SAME thread_id
# tells LangGraph to:
#   1. Load the last saved checkpoint for 'thread-1'
#   2. Check which node is pending next (step_2, since it crashed)
#   3. Resume execution from step_2 forward
#
# Result: step_1 does NOT re-run (it already completed and was saved).
# Only step_2 and step_3 execute — this time step_2 will complete normally
# because the hang only happens on the first call.
#
# In a real application: this pattern means you never lose completed work,
# even after server restarts, network failures, or OOM kills.
# ─────────────────────────────────────────────────────────────

print("\n🔁 Re-running the graph to demonstrate fault tolerance...")

# Resume from the last saved checkpoint for thread-1
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})

print("\n✅ Final State:", final_state)

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 26: Review the Full Checkpoint History After Recovery
# ─────────────────────────────────────────────────────────────
# The history for 'thread-1' shows checkpoints from BOTH runs:
#   - From Run 1 (crashed): checkpoint after step_1
#   - From Run 2 (recovery): checkpoints after step_2 and step_3
#
# This confirms that:
#   ✅ step_1's output was preserved through the crash
#   ✅ The recovery picked up from step_2, not from scratch
#   ✅ The complete execution chain is traceable via checkpoint history
# ─────────────────────────────────────────────────────────────

list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))